# Kuravisor Feasibility Study — Advanced Exploratory Data Analysis

**Research title:** Kuravisor: An Offline AI-Powered Crop Disease Detection and Treatment Recommendation System for Smallholder Farmers in Zimbabwe

**Theme:** Area 8 – Empowering SMEs with 4IR and AI for Inclusive and Sustainable Growth  
**Category:** Prototype Demonstration

---

This notebook evaluates whether the **Plant Leaf Diseases Dataset (with augmentation)** supports training an **offline, on-device CNN** for Kuravisor. Analyses cover:

1. Dataset inventory and class balance  
2. Alignment with **Zimbabwe smallholder crops** (maize, tomato, potato, pepper, squash, legumes)  
3. Image quality and mobile-camera suitability  
4. Intra-class visual diversity (augmentation / field variability)  
5. Stratified train–validation–test feasibility  
6. Offline inference constraints (resolution, model footprint)  
7. Disease vs. healthy taxonomy for advisory logic  
8. Research-oriented summary metrics for the paper

> **Dataset path:** `dataset/Plant_leave_diseases_dataset_with_augmentation/` (gitignored locally).  
> **Sampling:** Image-level EDA uses configurable random sampling for speed; set `SAMPLE_PER_CLASS = None` for a full pass.


In [ ]:
# --- Configuration ---
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from scipy import stats
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "dataset").exists():
    REPO_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "dataset").exists():
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = NOTEBOOK_DIR

DATASET_ROOT = REPO_ROOT / "dataset" / "Plant_leave_diseases_dataset_with_augmentation"
OUTPUT_DIR = REPO_ROOT / "notebooks" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG"}
SAMPLE_PER_CLASS = 80  # None = analyze all images
TARGET_INPUT_SIZE = (224, 224)

sns.set_theme(style="whitegrid", context="talk", palette="husl")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

print("Repository root:", REPO_ROOT)
print("Dataset root:", DATASET_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())


## 1. Dataset inventory

Build a class-level catalog: image counts, crop species, disease labels, and file formats.


In [ ]:
def parse_class_name(class_name: str) -> dict:
    parts = class_name.split("___", 1)
    crop = parts[0].replace("_", " ").strip() if parts else class_name
    condition = parts[1].replace("_", " ").strip() if len(parts) > 1 else "unknown"
    healthy = "healthy" in condition.lower()
    return {"class_name": class_name, "crop": crop, "condition": condition, "is_healthy": healthy}


def list_images(class_dir: Path):
    return [p for p in class_dir.iterdir() if p.is_file() and p.suffix in IMAGE_EXTENSIONS]


if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_ROOT}. "
        "Place Plant_leave_diseases_dataset_with_augmentation under dataset/."
    )

class_dirs = sorted([d for d in DATASET_ROOT.iterdir() if d.is_dir()])
rows = []
for d in tqdm(class_dirs, desc="Counting images"):
    meta = parse_class_name(d.name)
    imgs = list_images(d)
    rows.append({**meta, "n_images": len(imgs), "class_dir": str(d)})

df_classes = pd.DataFrame(rows)
df_classes["crop_key"] = df_classes["crop"].str.lower().str.replace(" ", "_")

total_images = df_classes["n_images"].sum()
n_classes = len(df_classes)
n_crops = df_classes["crop"].nunique()

print(f"Classes: {n_classes} | Crops: {n_crops} | Total images: {total_images:,}")
df_classes.sort_values("n_images", ascending=False).head(10)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

order = df_classes.sort_values("n_images", ascending=True)
colors = np.where(order["is_healthy"], "#2ecc71", "#e74c3c")
axes[0].barh(order["class_name"], order["n_images"], color=colors)
axes[0].set_xlabel("Number of images")
axes[0].set_title("Images per class (green = healthy)")
axes[0].tick_params(axis="y", labelsize=7)

crop_counts = df_classes.groupby("crop")["n_images"].sum().sort_values(ascending=True)
axes[1].barh(crop_counts.index, crop_counts.values, color="#3498db")
axes[1].set_xlabel("Total images")
axes[1].set_title("Images per crop species")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_class_and_crop_distribution.png", bbox_inches="tight")
plt.show()

df_classes.describe()[["n_images"]]


## 2. Zimbabwe smallholder crop relevance

Map dataset crops to **priority tiers** for Kuravisor based on Zimbabwe horticulture and staple production (maize dominant; tomato, potato, pepper, squash widely grown).

| Tier | Crops in dataset | Rationale |
|------|------------------|-----------|
| **Tier 1 – High** | Corn (maize), Tomato, Potato, Pepper (bell), Squash | Staple / top traded horticulture |
| **Tier 2 – Medium** | Soybean, Grape, Peach, Strawberry | Legumes / supplementary horticulture |
| **Tier 3 – Contextual** | Apple, Cherry, Orange, Blueberry, Raspberry | Lower smallholder prevalence |

*Maize appears as **Corn** in PlantVillage nomenclature.*


In [ ]:
ZIMBABWE_RELEVANCE = {
    "Corn": "Tier 1 – High (maize staple)",
    "Tomato": "Tier 1 – High",
    "Potato": "Tier 1 – High",
    "Pepper, bell": "Tier 1 – High",
    "Squash": "Tier 1 – High",
    "Soybean": "Tier 2 – Medium (legume)",
    "Grape": "Tier 2 – Medium",
    "Peach": "Tier 2 – Medium",
    "Strawberry": "Tier 2 – Medium",
    "Apple": "Tier 3 – Contextual",
    "Cherry": "Tier 3 – Contextual",
    "Orange": "Tier 3 – Contextual",
    "Blueberry": "Tier 3 – Contextual",
    "Raspberry": "Tier 3 – Contextual",
    "Background without leaves": "Non-target (filter at capture)",
}

df_classes["zimbabwe_tier"] = df_classes["crop"].map(
    lambda c: ZIMBABWE_RELEVANCE.get(c, "Unmapped")
)
df_classes["tier_group"] = df_classes["zimbabwe_tier"].str.extract(r"(Tier \d)")[0].fillna("Other")

tier_summary = (
    df_classes.groupby(["tier_group", "crop"])
    .agg(n_classes=("class_name", "count"), n_images=("n_images", "sum"))
    .reset_index()
    .sort_values(["tier_group", "n_images"], ascending=[True, False])
)
tier_summary


In [ ]:
tier_totals = df_classes.groupby("tier_group").agg(
    classes=("class_name", "count"),
    images=("n_images", "sum"),
).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(tier_totals))
ax.bar(x, tier_totals["images"], color=["#27ae60", "#f39c12", "#95a5a6", "#7f8c8d"][: len(tier_totals)])
ax.set_xticks(x)
ax.set_xticklabels(tier_totals["tier_group"])
ax.set_ylabel("Image count")
ax.set_title("Dataset volume by Zimbabwe relevance tier")
for i, row in tier_totals.iterrows():
    ax.text(i, row["images"] + 200, f"{row['classes']} classes", ha="center", fontsize=10)
plt.savefig(OUTPUT_DIR / "02_zimbabwe_tier_volume.png", bbox_inches="tight")
plt.show()

tier1 = df_classes[df_classes["tier_group"] == "Tier 1"]
tier1_pct = tier1["n_images"].sum() / total_images
print(f"Tier 1 (priority) images: {tier1['n_images'].sum():,} ({tier1_pct:.1%} of dataset)")


## 3. Class imbalance and stratification risk

Highly imbalanced classes can bias CNN training. We report **Gini coefficient**, **entropy**, and **min–max spread**.


In [ ]:
counts = df_classes["n_images"].values
proportions = counts / counts.sum()

sorted_c = np.sort(counts)
n = len(counts)
gini_coef = (2 * np.sum((np.arange(1, n + 1) * sorted_c)) / (n * sorted_c.sum())) - (n + 1) / n
entropy = stats.entropy(proportions)
imbalance_ratio = counts.max() / counts.min()

metrics_imbalance = pd.DataFrame([{
    "n_classes": n_classes,
    "min_images_per_class": int(counts.min()),
    "max_images_per_class": int(counts.max()),
    "mean_images_per_class": float(counts.mean()),
    "std_images_per_class": float(counts.std()),
    "imbalance_ratio_max_min": float(imbalance_ratio),
    "gini_coefficient": float(gini_coef),
    "shannon_entropy_bits": float(entropy),
}])
metrics_imbalance.T


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_classes["n_images"], bins=20, kde=True, ax=axes[0], color="#8e44ad")
med = df_classes["n_images"].median()
axes[0].axvline(med, color="red", ls="--", label=f"median={med:.0f}")
axes[0].set_xlabel("Images per class")
axes[0].set_title("Distribution of class sizes")
axes[0].legend()

cum_share = np.cumsum(np.sort(counts)) / counts.sum()
axes[1].plot(np.linspace(0, 1, len(cum_share)), cum_share, lw=2)
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5)
axes[1].set_xlabel("Fraction of classes (sorted by size)")
axes[1].set_ylabel("Cumulative share of images")
axes[1].set_title("Concentration of samples across classes")
plt.savefig(OUTPUT_DIR / "03_class_imbalance.png", bbox_inches="tight")
plt.show()


## 4. Image-level quality analysis (mobile suitability)

Assess resolution, aspect ratio, file size, brightness, and sharpness for **offline smartphone inference**.


In [ ]:
def sample_image_paths(df_cls, per_class=SAMPLE_PER_CLASS, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    paths, labels = [], []
    for _, row in df_cls.iterrows():
        imgs = list_images(Path(row["class_dir"]))
        if not imgs:
            continue
        if per_class is None:
            chosen = imgs
        else:
            k = min(per_class, len(imgs))
            idx = rng.choice(len(imgs), size=k, replace=False)
            chosen = [imgs[i] for i in idx]
        paths.extend(chosen)
        labels.extend([row["class_name"]] * len(chosen))
    return paths, labels


def laplacian_variance(gray: np.ndarray) -> float:
    return float(gray.var())


def analyze_image(path: Path) -> dict:
    try:
        with Image.open(path) as im:
            im_rgb = im.convert("RGB")
            w, h = im_rgb.size
            arr = np.asarray(im_rgb.resize(TARGET_INPUT_SIZE, Image.BILINEAR)) / 255.0
        gray = 0.299 * arr[:, :, 0] + 0.587 * arr[:, :, 1] + 0.114 * arr[:, :, 2]
        return {
            "path": str(path),
            "width": w,
            "height": h,
            "aspect_ratio": w / h if h else np.nan,
            "file_size_kb": path.stat().st_size / 1024,
            "mean_brightness": float(gray.mean()),
            "std_brightness": float(gray.std()),
            "sharpness_laplacian_var": laplacian_variance(gray),
            "error": None,
        }
    except Exception as e:
        return {"path": str(path), "error": str(e)}


paths, labels = sample_image_paths(df_classes)
print(f"Sampled {len(paths):,} images (SAMPLE_PER_CLASS={SAMPLE_PER_CLASS})")

records = []
for p, lab in tqdm(zip(paths, labels), total=len(paths), desc="Analyzing images"):
    rec = analyze_image(Path(p))
    rec["class_name"] = lab
    records.append(rec)

df_images = pd.DataFrame(records)
df_images = df_images[df_images["error"].isna()].drop(columns=["error"])
df_images = df_images.merge(
    df_classes[["class_name", "crop", "tier_group", "is_healthy"]], on="class_name", how="left"
)
print(f"Valid samples: {len(df_images):,}")
df_images.head()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.scatterplot(data=df_images, x="width", y="height", hue="tier_group", alpha=0.5, ax=axes[0, 0], legend=False)
axes[0, 0].set_title("Native resolution scatter")
axes[0, 0].axhline(TARGET_INPUT_SIZE[1], color="gray", ls=":", alpha=0.7)
axes[0, 0].axvline(TARGET_INPUT_SIZE[0], color="gray", ls=":", alpha=0.7)

sns.histplot(df_images["aspect_ratio"], bins=30, ax=axes[0, 1], color="#16a085")
axes[0, 1].set_xlabel("Aspect ratio (width/height)")

sns.boxplot(data=df_images, x="tier_group", y="mean_brightness", ax=axes[1, 0])
axes[1, 0].set_title("Brightness by relevance tier")
axes[1, 0].tick_params(axis="x", rotation=15)

sns.boxplot(data=df_images, x="tier_group", y="sharpness_laplacian_var", ax=axes[1, 1])
axes[1, 1].set_title("Sharpness proxy by tier")
axes[1, 1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "04_image_quality_mobile.png", bbox_inches="tight")
plt.show()

pct_above_224 = ((df_images["width"] >= 224) & (df_images["height"] >= 224)).mean()
print(f"Images with both sides >= 224px: {pct_above_224:.1%}")
print(f"Median resolution: {df_images['width'].median():.0f} x {df_images['height'].median():.0f}")
print(f"Median file size: {df_images['file_size_kb'].median():.1f} KB")


## 5. Intra-class visual diversity

Within-class variance indicates augmented or heterogeneous captures—relevant for **farmer field photos**.


In [ ]:
rgb_stats = (
    df_images.groupby("class_name")
    .agg(
        n_sampled=("path", "count"),
        brightness_mean=("mean_brightness", "mean"),
        brightness_std=("mean_brightness", "std"),
        sharpness_mean=("sharpness_laplacian_var", "mean"),
        sharpness_std=("sharpness_laplacian_var", "std"),
    )
    .reset_index()
    .merge(df_classes[["class_name", "crop", "tier_group", "n_images"]], on="class_name")
)
rgb_stats["brightness_cv"] = rgb_stats["brightness_std"] / (rgb_stats["brightness_mean"] + 1e-6)

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=rgb_stats,
    x="brightness_cv",
    y="sharpness_std",
    hue="tier_group",
    size="n_images",
    sizes=(40, 400),
    ax=ax,
    alpha=0.8,
)
ax.set_xlabel("Brightness CV (within class)")
ax.set_ylabel("Sharpness std (within class)")
ax.set_title("Intra-class heterogeneity")
plt.savefig(OUTPUT_DIR / "05_intra_class_diversity.png", bbox_inches="tight")
plt.show()

rgb_stats.sort_values("brightness_cv", ascending=False).head(8)


## 6. Train / validation / test split simulation

Stratified 70/15/15 split—minimum per-class counts for **prototype evaluation**.


In [ ]:
all_paths, all_labels = [], []
for _, row in df_classes.iterrows():
    for p in list_images(Path(row["class_dir"])):
        all_paths.append(str(p))
        all_labels.append(row["class_name"])

print("Total indexed images:", len(all_paths))

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30, stratify=all_labels, random_state=RANDOM_SEED
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, stratify=temp_labels, random_state=RANDOM_SEED
)

split_df = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n_images": [len(train_paths), len(val_paths), len(test_paths)],
})
split_df["pct"] = (split_df["n_images"] / split_df["n_images"].sum() * 100).round(1)
split_df


In [ ]:
def split_class_counts(labels):
    return pd.Series(labels).value_counts()

train_counts = split_class_counts(train_labels)
val_counts = split_class_counts(val_labels)
test_counts = split_class_counts(test_labels)

split_min = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "min_samples_per_class": [train_counts.min(), val_counts.min(), test_counts.min()],
    "classes_with_lt_10": [
        (train_counts < 10).sum(),
        (val_counts < 10).sum(),
        (test_counts < 10).sum(),
    ],
})
split_min


In [ ]:
weak_classes = train_counts[train_counts < 20]
print(f"Classes with <20 training images: {len(weak_classes)}")
if len(weak_classes):
    display(weak_classes.sort_values())


## 7. Offline on-device CNN feasibility indicators

Mobile architecture footprints and storage estimates for **TFLite deployment**.


In [ ]:
MODEL_ESTIMATES = pd.DataFrame([
    {"architecture": "MobileNetV2 (alpha=0.35)", "params_M": 1.7, "typical_tflite_MB": 2.5},
    {"architecture": "MobileNetV3-Small", "params_M": 2.5, "typical_tflite_MB": 3.0},
    {"architecture": "EfficientNet-Lite0", "params_M": 4.7, "typical_tflite_MB": 5.5},
    {"architecture": "Custom CNN (5-block)", "params_M": 1.2, "typical_tflite_MB": 1.5},
])
MODEL_ESTIMATES["feasible_offline"] = MODEL_ESTIMATES["typical_tflite_MB"] < 10
MODEL_ESTIMATES

n_train = len(train_paths)
bytes_per_sample = 224 * 224 * 3 * 4
gb_raw_epoch = n_train * bytes_per_sample / 1e9

feasibility = pd.DataFrame([{
    "n_classes": n_classes,
    "n_train_images": n_train,
    "recommended_input": f"{TARGET_INPUT_SIZE[0]}x{TARGET_INPUT_SIZE[1]}",
    "pct_images_ge_224px": round(float(pct_above_224) * 100, 1),
    "est_raw_epoch_GB_float32": round(gb_raw_epoch, 2),
    "target_tflite_size_MB": "< 10",
    "inference_mode": "On-device, no connectivity",
}])
feasibility.T


## 8. Disease taxonomy for Kuravisor advisory logic

**Healthy**, **diseased**, and **background** scopes for treatment modules and alerts.


In [ ]:
df_classes["label_type"] = np.select(
    [
        df_classes["class_name"].str.contains("Background", case=False),
        df_classes["is_healthy"],
    ],
    ["background", "healthy"],
    default="diseased",
)

taxonomy = df_classes.groupby("label_type").agg(
    classes=("class_name", "count"),
    images=("n_images", "sum"),
).reset_index()
taxonomy["pct_images"] = (taxonomy["images"] / taxonomy["images"].sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(taxonomy["images"], labels=taxonomy["label_type"], autopct="%1.1f%%", startangle=140)
ax.set_title("Healthy vs diseased vs background")
plt.savefig(OUTPUT_DIR / "06_taxonomy_pie.png", bbox_inches="tight")
plt.show()
taxonomy


## 9. Visual audit — Tier 1 (Zimbabwe priority) crops

Qualitative montages for maize, tomato, potato, pepper, and squash.


In [ ]:
def show_class_montage(class_name, n=4, seed=RANDOM_SEED):
    row = df_classes[df_classes["class_name"] == class_name].iloc[0]
    imgs = list_images(Path(row["class_dir"]))
    rng = np.random.default_rng(seed)
    chosen = rng.choice(imgs, size=min(n, len(imgs)), replace=False)
    fig, axes = plt.subplots(1, len(chosen), figsize=(3 * len(chosen), 3))
    if len(chosen) == 1:
        axes = [axes]
    for ax, p in zip(axes, chosen):
        ax.imshow(Image.open(p).convert("RGB"))
        ax.axis("off")
    fig.suptitle(class_name, fontsize=11)
    plt.tight_layout()
    plt.show()


for crop in df_classes[df_classes["tier_group"] == "Tier 1"]["crop"].unique():
    subset = df_classes[df_classes["crop"] == crop]
    for _, row in subset.iterrows():
        show_class_montage(row["class_name"], n=4)


## 10. Feasibility conclusions and research metrics export

Summary for the Kuravisor paper and SDG alignment.


In [ ]:
tier1_images = int(df_classes.loc[df_classes["tier_group"] == "Tier 1", "n_images"].sum())
tier1_classes = int(df_classes.loc[df_classes["tier_group"] == "Tier 1", "class_name"].nunique())
diseased_images = int(df_classes.loc[df_classes["label_type"] == "diseased", "n_images"].sum())

conclusions = {
    "study": "Kuravisor feasibility EDA",
    "dataset": DATASET_ROOT.name,
    "total_images": int(total_images),
    "total_classes": int(n_classes),
    "total_crops": int(n_crops),
    "tier1_zimbabwe_priority_images": tier1_images,
    "tier1_pct_of_dataset": round(tier1_images / total_images * 100, 1),
    "tier1_classes": tier1_classes,
    "diseased_class_images": diseased_images,
    "gini_coefficient": round(float(gini_coef), 4),
    "imbalance_ratio_max_min": round(float(imbalance_ratio), 2),
    "train_split_images": int(len(train_paths)),
    "min_train_samples_per_class": int(train_counts.min()),
    "classes_train_lt_20": int((train_counts < 20).sum()),
    "sampled_images_quality_eda": int(len(df_images)),
    "median_resolution": f"{int(df_images['width'].median())}x{int(df_images['height'].median())}",
    "pct_images_ge_224px": round(float(pct_above_224) * 100, 1),
    "feasibility_offline_cnn": "Supported — sufficient labeled imagery; Tier 1 fine-tuning + Zimbabwe field validation required",
    "feasibility_multilingual_ui": "Requires trilingual treatment knowledge base (EN/Sn/Nd)",
    "feasibility_predictive_alerts": "Needs environmental/time-series data beyond static images",
    "limitations": "PlantVillage domain shift; no local variety labels; extension worker validation",
    "sdg_alignment": "SDG 2 (Zero Hunger), SDG 9 (Innovation/Infrastructure)",
}

summary_df = pd.DataFrame([conclusions]).T.rename(columns={0: "value"})
display(summary_df)

summary_df.to_csv(OUTPUT_DIR / "kuravisor_feasibility_summary.csv", header=["value"])
df_classes.to_csv(OUTPUT_DIR / "class_catalog.csv", index=False)
rgb_stats.to_csv(OUTPUT_DIR / "intra_class_diversity.csv", index=False)
metrics_imbalance.to_csv(OUTPUT_DIR / "imbalance_metrics.csv", index=False)
print("Exported metrics to", OUTPUT_DIR)


---

### Interpretation guide for the research paper

| Finding | Implication for Kuravisor |
|--------|---------------------------|
| Large Tier 1 volume | Strong baseline for Zimbabwe-relevant CNN classes |
| Class imbalance | Use class weights / focal loss; report macro-F1 |
| Images ≥ 224px | Compatible with MobileNet / EfficientNet-Lite |
| High intra-class brightness CV | Plan augmentation + mandatory field testing |
| Stratified split minima | Validate per-class recall before treatment advice |
| Background class | UX: leaf should fill the frame |

**Next steps:** Tier 1 fine-tuning, Zimbabwe field image collection, trilingual treatment KB, TFLite latency on target devices, co-design with extension workers.
